In [1]:
import os
from datasets import Dataset
from torchvision.transforms import Compose, Resize, ToTensor, Grayscale, Lambda
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image

dir = "data"
dataset_name = "SID_saved"
dataset_dir = os.path.join(dir, dataset_name)

c:\Users\Bjark\miniconda3\envs\nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from datasets import load_from_disk
loaded_dataset = load_from_disk(dataset_dir)

In [3]:
# 1. Create a transform pipeline
image_transform = Compose([
    # Force convert any image to RGB (3 channels)
    Resize((256, 256)), # resize to fixed size
    Lambda(lambda img: img.convert("RGB") if isinstance(img, Image.Image) else img), # Some of the images are grayscale
    ToTensor()   # now always yields shape (3,256,256)
])

mask_transform = Compose([
    Resize((256, 256)),
    Grayscale(num_output_channels=1),  # force 1 channel
    ToTensor()                         # yields shape (1,256,256)
])

class HFDataset(Dataset):
    def __init__(self, hf_ds, device=None):
        self.ds = hf_ds
        self.device = device

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        ex = self.ds[idx]

        # Transform image
        img = image_transform(ex["image"])  # → tensor shape (3,256,256)
        #print(f"Image shape: {img.shape}, Type: {type(img)}, Dtype: {img.dtype}, Min: {img.min().item()}, Max: {img.max().item()}")

        # Transform mask if present
        if ex["mask"] is not None:
            mask = mask_transform(ex["mask"])
        else:
            mask = torch.zeros((1,256,256), dtype=torch.float)
        #print(f"Mask shape: {mask.shape}, Type: {type(mask)}, Dtype: {mask.dtype}, Min: {mask.min().item()}, Max: {mask.max().item()}")

        label = torch.tensor(ex["label"], dtype=torch.long)
        #print(f"Label: {label}, Type: {type(label)}, Value: {label.item()}")

        if self.device:
            img, mask, label = img.to(self.device), mask.to(self.device), label.to(self.device)
        return {"image": img, "mask": mask, "label": label}

device = torch.device("cuda") if torch.cuda.is_available() else None
wrapped = HFDataset(loaded_dataset, device=device)

loader = DataLoader(
    wrapped,
    batch_size=20,
    shuffle=True,
    num_workers=0,     # adjust based on your CPU cores
    pin_memory=bool(device)
)
batch = next(iter(loader))
print(f"Batch of images: {batch['image'].shape}, Batch of masks: {batch['mask'].shape}, Batch of labels: {batch['label'].shape}")


batch = next(iter(loader))
print(f"Batch of images: {batch['image'].shape}")  # → (batch_size,3,256,256)
print(f"Batch of masks: {batch['mask'].shape}")   # → (batch_size,1,256,256)


Batch of images: torch.Size([20, 3, 256, 256]), Batch of masks: torch.Size([20, 1, 256, 256]), Batch of labels: torch.Size([20])
Batch of images: torch.Size([20, 3, 256, 256])
Batch of masks: torch.Size([20, 1, 256, 256])
